# Visualize Virtual Update
## Swiss Roll Dataset

In [ ]:
from torch.optim import SGD
from torch.nn import MSELoss, ReLU
import torch

from sklearn.datasets import make_swiss_roll
from matplotlib import pyplot as plt
import sys

sys.path.append('..')
from src.data import CustomDataset
from src.model import DNN

In [ ]:
X, y = make_swiss_roll(n_samples=1000, random_state=0)
X = torch.tensor([X[:, 0], X[:, 2]]).T
X = X - X.min(dim=0).values
X = X / X.max(dim=0).values

y = torch.tensor(y)
y = y - y.min(dim=0).values
y = y / y.max(dim=0).values

X = X.to(torch.float)
y = y.to(torch.float)

In [ ]:
plt.scatter(X[:, 0], X[:, 1], c=y)

In [ ]:
model = DNN(input_dim=2, hidden_dim=32, num_hidden=2, output_dim=1, bias=True)
model.layers[0].bias = None
optim = SGD(model.parameters(), lr=0.05)
loss_fn = MSELoss()
loss_list = list()
input_grad_list = list()
weight_list = list()

X.requires_grad_(True)
weight_list.append(model.layers[0].weight.data.clone())

for i in range(100):
    optim.zero_grad()
    pred = model(X)
    loss = loss_fn(pred, y.view(-1, 1))
    loss.backward(retain_graph=True)
    grad = torch.autograd.grad(loss, X, retain_graph=True)[0]
    optim.step()
    loss_list.append(loss.item())
    input_grad_list.append(grad.clone())
    weight_list.append(model.layers[0].weight.data.clone())

print(" -> ".join([str(round(i, 3)) for i in loss_list]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import multivariate_normal
import torch

plot_lim = (-0.2, 1.2)
frames = [0, 10, 20, 30, 40, 50]

grid_pts = np.linspace(plot_lim[0], plot_lim[1], 50)
grid_x, grid_y = np.meshgrid(grid_pts, grid_pts)
pos = np.dstack((grid_x, grid_y))
X = X.detach()
X_mean = X.mean(dim=0)

fig, axes = plt.subplots(1, len(frames), figsize=(3.5 * len(frames), 3.8))

for i, (ax, frame) in enumerate(zip(axes, frames)):
    ax.set_xlim(plot_lim)
    ax.set_ylim(plot_lim)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])

    # Original X (background)
    h_orig = ax.scatter(X[:, 0], X[:, 1], c='gray', alpha=0.15, s=60, label='Original $X$')

    # Virtual Xp
    if frame == 0:
        Xp = X
    else:
        Xp = X - 7e+2 * torch.stack(input_grad_list[:frame + 1]).sum(dim=0)
        Xp = Xp - Xp.min(dim=0).values
        Xp = Xp / (Xp.max(dim=0).values + 1e-6)

    h_virt = ax.scatter(Xp[:, 0], Xp[:, 1], c=y, cmap='viridis', s=70, label='Virtual $\\tilde{X}$')

    # Gram metric contour — solid lines for visibility at small size
    current_weight = weight_list[frame]
    current_gram = (current_weight.T @ current_weight).detach().numpy()
    current_gram += np.eye(2) * 1e-3

    rv = multivariate_normal(X_mean.numpy(), current_gram)
    ax.contour(grid_x, grid_y, rv.pdf(pos), levels=6,
               colors='red', alpha=0.75, linestyles='dashed', linewidths=2.0)

    ax.set_title(f"Epoch {frame}", fontsize=20, fontweight='bold', pad=6)

    # Legend on first subplot only
    if i == 5:
        contour_proxy = plt.Line2D([0], [0], color='red', alpha=0.75,
                                   linewidth=5, label='Metric $W^\\top W$')
        ax.legend(handles=[h_orig, h_virt, contour_proxy],
                  loc='lower right', fontsize=14,
                  markerscale=2.0, framealpha=0.9,
                  handlelength=1.5, borderpad=0.6)

fig.tight_layout()
plt.savefig("../plot/swissroll_virtual_update.png", dpi=200, bbox_inches='tight')
plt.show()